# Notebook 2: Perceptron with NumPy — Vectorised Operations

In the previous notebook we built a perceptron using pure Python loops. That's great for understanding, but real neural networks use **vectorised math** for speed.

In this notebook you will:

1. Learn what a **dot product** is and why it matters
2. Re-implement the perceptron using **NumPy** arrays
3. See how vector operations replace explicit loops
4. Compare the pure-Python and NumPy approaches side by side

---

## 1. Quick Recap — The Dot Product

The **dot product** of two vectors is the sum of element-wise multiplications:

$$\mathbf{w} \cdot \mathbf{x} = w_1 x_1 + w_2 x_2 + w_3 x_3$$

This is exactly the weighted sum we computed manually in Notebook 1! NumPy's `np.dot()` does this in a single, optimised call.

In [1]:
import numpy as np

# Example: manual vs. dot product
w = np.array([0.5, -0.3, 0.8])
x = np.array([1, 1, 0])

manual = w[0]*x[0] + w[1]*x[1] + w[2]*x[2]
dot    = np.dot(w, x)

print(f"Manual calculation: {manual}")
print(f"np.dot() result:   {dot}")
print(f"Are they equal?    {np.isclose(manual, dot)}")

Manual calculation: 0.2
np.dot() result:   0.2
Are they equal?    True


## 2. Initialise Weights with NumPy

Instead of a Python list we use a NumPy array. This lets us do arithmetic on the whole vector at once.

In [2]:
np.random.seed(42)  # reproducibility

weights = np.random.uniform(-1, 1, size=3)   # 3 random weights
bias = np.random.uniform(-1, 1)
learning_rate = 0.1

print("Starting weights:", np.round(weights, 3))
print("Starting bias:   ", round(bias, 3))

Starting weights: [-0.251  0.901  0.464]
Starting bias:    0.197


## 3. Training Data as NumPy Arrays

In [3]:
training_data = [
    # Vehicles (label = 1)
    (np.array([1, 1, 0]), 1),  # Car
    (np.array([0, 1, 0]), 1),  # Truck
    (np.array([1, 1, 0]), 1),  # Bicycle
    (np.array([0, 1, 0]), 1),  # Bus
    # Living Beings (label = 0)
    (np.array([1, 0, 1]), 0),  # Human
    (np.array([0, 0, 1]), 0),  # Dog
    (np.array([0, 0, 1]), 0),  # Cat
    (np.array([1, 0, 1]), 0),  # Bird
]

print(f"{len(training_data)} training examples loaded.")

8 training examples loaded.


## 4. Training — Pure Python vs. NumPy Side-by-Side

The key differences in the training loop are highlighted below:

| Step | Pure Python (Notebook 1) | NumPy (this notebook) |
|------|-------------------------|----------------------|
| Weighted sum | `w[0]*x[0] + w[1]*x[1] + w[2]*x[2] + bias` | `np.dot(weights, features) + bias` |
| Weight update | Loop over each weight | `weights += lr * error * features` |

That's it — the algorithm is identical, only the implementation changes.

In [4]:
max_epochs = 100
history = []

print("Training the Perceptron (NumPy)...")
print("-" * 60)

for epoch in range(max_epochs):
    errors = 0

    for features, target in training_data:
        # ---- Forward pass (dot product!) ----
        total = np.dot(weights, features) + bias
        prediction = 1 if total >= 0 else 0

        # ---- Error & update ----
        error = target - prediction
        if error != 0:
            errors += 1
            # Vectorised weight update — no loop!
            weights = weights + learning_rate * error * features
            bias = bias + learning_rate * error

    history.append(errors)
    print(f"Epoch {epoch + 1:3d} | Errors: {errors} | "
          f"Weights: {np.round(weights, 3)} | Bias: {bias:.3f}")

    if errors == 0:
        print(f"\n>>> Training converged in {epoch + 1} epoch(s).")
        break

Training the Perceptron (NumPy)...
------------------------------------------------------------
Epoch   1 | Errors: 3 | Weights: [-0.351  0.901  0.164] | Bias: -0.103
Epoch   2 | Errors: 1 | Weights: [-0.351  0.901  0.064] | Bias: -0.203
Epoch   3 | Errors: 0 | Weights: [-0.351  0.901  0.064] | Bias: -0.203

>>> Training converged in 3 epoch(s).


## 5. Testing

In [5]:
test_cases = [
    (np.array([1, 1, 0]), "Mystery 1 — round, wheels, no breathing"),
    (np.array([0, 0, 1]), "Mystery 2 — not round, no wheels, breathes"),
    (np.array([1, 0, 1]), "Mystery 3 — round, no wheels, breathes"),
    (np.array([0, 1, 0]), "Mystery 4 — not round, wheels, no breathing"),
]

print("=" * 55)
print("Testing the Perceptron")
print("=" * 55)

for features, description in test_cases:
    total = np.dot(weights, features) + bias
    prediction = 1 if total >= 0 else 0
    label = "VEHICLE" if prediction == 1 else "LIVING BEING"

    print(f"\n{description}")
    print(f"  Dot product + bias = {total:.3f}")
    print(f"  Prediction: {label}")

Testing the Perceptron

Mystery 1 — round, wheels, no breathing
  Dot product + bias = 0.348
  Prediction: VEHICLE

Mystery 2 — not round, no wheels, breathes
  Dot product + bias = -0.139
  Prediction: LIVING BEING

Mystery 3 — round, no wheels, breathes
  Dot product + bias = -0.490
  Prediction: LIVING BEING

Mystery 4 — not round, wheels, no breathing
  Dot product + bias = 0.699
  Prediction: VEHICLE


## 6. Understanding the Dot Product — A Deeper Look

The dot product measures how **aligned** two vectors are. Let's build some intuition.

In [6]:
# The learned weights form a "template" for what a vehicle looks like.
# The dot product tells us how similar an input is to that template.

print("Learned weight vector:", np.round(weights, 3))
print()

# Compare a clear vehicle vs. a clear living being
vehicle      = np.array([1, 1, 0])  # car
living_being = np.array([0, 0, 1])  # dog

print(f"Dot product with vehicle {vehicle}:        {np.dot(weights, vehicle):.3f}")
print(f"Dot product with living being {living_being}:  {np.dot(weights, living_being):.3f}")
print()
print("Notice how the vehicle gets a HIGHER score (more aligned with the weight vector).")
print("The bias then shifts the decision boundary so the step function separates them.")

Learned weight vector: [-0.351  0.901  0.064]

Dot product with vehicle [1 1 0]:        0.551
Dot product with living being [0 0 1]:  0.064

Notice how the vehicle gets a HIGHER score (more aligned with the weight vector).
The bias then shifts the decision boundary so the step function separates them.


## 7. Showing the Mathematical Equivalence

Let's verify that the dot product is just shorthand for what we did manually in Notebook 1.

In [7]:
test_features = np.array([1, 1, 0])  # a car

# Method 1: Manual (pure Python style)
manual_sum = weights[0]*test_features[0] + weights[1]*test_features[1] + weights[2]*test_features[2]

# Method 2: Dot product
dot_sum = np.dot(weights, test_features)

print("The dot product is just a shorthand for:")
print(f"  weights . features = {weights[0]:.3f}*{test_features[0]} + {weights[1]:.3f}*{test_features[1]} + {weights[2]:.3f}*{test_features[2]}")
print(f"")
print(f"  Manual calculation: {manual_sum:.3f}")
print(f"  np.dot() result:   {dot_sum:.3f}")
print(f"  Match: {np.isclose(manual_sum, dot_sum)}")

The dot product is just a shorthand for:
  weights . features = -0.351*1 + 0.901*1 + 0.064*0

  Manual calculation: 0.551
  np.dot() result:   0.551
  Match: True


## 8. Why NumPy Matters at Scale

With 3 features and 8 examples the speed difference is negligible. But modern language models have **billions** of parameters. Vectorised operations run on optimised C/Fortran and can be parallelised on GPUs — making them orders of magnitude faster.

| Approach | 3 features | 1,000 features | 1,000,000 features |
|----------|-----------|----------------|--------------------|
| Python loop | fast | slow | very slow |
| NumPy dot | fast | fast | fast |
| GPU (PyTorch/TF) | fast | fast | **very** fast |

## 9. Exercises

1. **Batch all features into a matrix.** Stack all training features into a 2-D array `X` of shape `(8, 3)` and labels into a 1-D array `y`. Rewrite the training loop so the forward pass for all samples is a single `X @ weights + bias`.
2. **Time it.** Use `%%timeit` to compare the pure-Python loop (Notebook 1) with the NumPy version on 10,000 random samples.
3. **Visualise alignment.** For each test case, print the angle between the weight vector and the feature vector using `np.arccos(dot / (|w| * |x|))`.

---

## 10. Summary

| Concept | What we learned |
|---------|----------------|
| Dot product | Replaces the manual weighted sum |
| Vectorised update | `weights += lr * error * features` — no loop |
| NumPy arrays | Foundation for all scientific Python |
| Scalability | Vectorised code scales to millions of parameters |

**Next:** Open *Notebook 3* to **visualise** the perceptron's decision boundary and training dynamics.